In [84]:
"""
====================================================
Download ERA5 Data
====================================================
"""

'\n====================================================\nDownload ERA5 Data\n====================================================\n'

In [85]:
#######################
# DIRECTORIES

In [86]:
#SETTING UP DIRECOTRIES
mainDirectory = "/glade/u/home/aroseman/Projects/Regional-MPAS-Project/Code/CodeFiles/"
outputDirectory=mainDirectory+"../DATA/ERA5_Data/"
import os; os.makedirs(outputDirectory, exist_ok=True)

In [87]:
#######################
# LIBRARIES, FUNCTIONS, and CLASSES

In [88]:
# IMPORT LIBRARIES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "/Libraries/"
sys.path.append(path)

# --- Import all your function modules ---
import importlib

modules = [
    "Libraries",
]

for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [89]:
# IMPORT FUNCTIONS
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "AreaAverageFunctions",
    "ComputationFunctions",
    "DataFunctions",
    "DerivativeFunctions",
    "PlottingFunctions",
    "StatisticalFunctions",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [90]:
# IMPORT CLASSES
# --- Add your Functions folder to sys.path ---
import sys

path = mainDirectory + "Functions_2.0/Classes/"
sys.path.append(path)


# --- Import all your function modules ---
import importlib

modules = [
    "Classes_InputData_DataAnalysis",
]
for mod in modules:
    globals()[mod] = importlib.import_module(mod)  # import module itself
    globals().update(vars(globals()[mod]))  # import all functions into global namespace

In [91]:
###########################
# DOWNLOADING DATA FUNCTIONS

In [92]:
# DOWNLOADING ERA5 (Pressure Levels)
# Code Inspired from "Download_ERA5_with_python" by github.com/joaohenry23 at https://github.com/joaohenry23/Download_ERA5_with_python

#PRELIMINARY STEPS
#(1) go to https://cds.climate.copernicus.eu/how-to-api
#(2) make file in main user directory called .cdsapirc
#(3) copy the following into file: 
#    url: https://cds.climate.copernicus.eu/api
#    key: 6d55399f-dbc5-48bd-848c-31168fc0b133 (key will be different for you based on your account)
#(4) pip install "cdsapi>=0.7.4"

import cdsapi 
def DownloadERA5_PressureLevels(variables, date, area):
    c = cdsapi.Client()
    for variable in tqdm(variables, desc="Downloading ERA5 variables"):
        print(f"Downloading {variable}", "\n")
        c.retrieve(
            "reanalysis-era5-pressure-levels",
            {
                "product_type": "reanalysis",
                "format": "netcdf",
                "variable": variable,
                # "pressure_level": ['100', '250', '500', '750', '1000'], #LOW-RES
                "pressure_level": [
                    '10', '20', '30', '50', '70', 
                    '100', '125', '150', '175', '200', '225',
                    '250', '300', '350', '400', '450', '500',
                    '550', '600', '650', '700', '750', '775',
                    '800', '825', '850', '875', '900', '925',
                    '950', '975', '1000',
                ],

                "date": date,
                "time": [f"{h:02d}:00" for h in range(24)],
                "area": area,
                "grid": [0.25, 0.25],
            },
            os.path.join(
                date_directory, f"{variable}_ERA5_{date_folder}.nc"
            ),
        )

#EXAMPLE RUN
# date_string = "06-30 - 07-02 (2022)"
# date_folder = MakeDateFolder(date_string)
# date_string_converted = date_string_to_range(date_string)

# # running
# DownloadERA5_PressureLevels(variables,date_string_converted, area)

In [93]:
# DOWNLOADING ERA5 (Surface)

#PRELIMINARY STEPS
#(1) go to https://cds.climate.copernicus.eu/how-to-api
#(2) make file in main user directory called .cdsapirc
#(3) copy the following into file: 
#    url: https://cds.climate.copernicus.eu/api
#    key: 6d55399f-dbc5-48bd-848c-31168fc0b133 (key will be different for you based on your account)
#(4) pip install "cdsapi>=0.7.4"

import cdsapi 
def DownloadERA5_Surface(variables, date, area):
    c = cdsapi.Client()
    for variable in tqdm(variables, desc="Downloading ERA5 surface variables"):
        print(f"Downloading {variable}", "\n")
        c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "format": "netcdf",
                "variable": variable,
                "date": date,  # e.g. "2022-06-08/2022-06-10"
                "time": [f"{h:02d}:00" for h in range(24)],  # every hour
                "area": area,  # [North, West, South, East]
                "grid": [0.25, 0.25],
            },
            os.path.join(
                date_directory, f"{variable}_ERA5_{date_folder}.nc"
            ),
        )

#EXAMPLE RUN
# date_string = "06-30 - 07-02 (2022)"
# date_folder = MakeDateFolder(date_string)
# date_string_converted = date_string_to_range(date_string)

# # running
# DownloadERA5_Surface(variables,date_string_converted, area)

In [102]:
# DATE INFORMATION
def date_string_to_range(date_string: str) -> str:
    """
    Convert a date string like "06-30 - 07-02 (2022)"
    into ERA5 API format: "2022-06-30/to/2022-07-02".
    """
    # Extract year
    year = date_string.split("(")[1].replace(")", "").strip()

    # Extract the two parts safely
    date_part = date_string.split("(")[0].strip()  # "06-30 - 07-02"
    start, end = date_part.split(" - ")            # ["06-30", "07-02"]

    # Make full YYYY-MM-DD
    start_date = f"{year}-{start}"
    end_date   = f"{year}-{end}"

    return f"{start_date}/{end_date}"

    
def MakeDateFolder(date_string, campaign):
    date_folder = strings.DateString(date_string)
    # adding date to output folder
    date_directory = os.path.join(outputDirectory, campaign, date_folder)
    os.makedirs(date_directory, exist_ok=True)
    return date_directory, date_folder


# COORDINATES INFORMATION
def GetCoordinates(longitude, latitude, dx_m=250e3, dy_m=250e3, grid_res=0.25, latlon_type='decimal'):

    # if not in decimal form use 
    if latlon_type != "decimal":
        #e.g. longitude = (95, 17, 2, "W"); latitude = (29, 31, 55, "N")
        longitude = coordinates.DMSToDecimal(*longitude)
        latitude = coordinates.DMSToDecimal(*latitude)

    dlon = coordinates.dxTOdlon(dx_m=dx_m, lat_deg=latitude)
    dlat = coordinates.dyTOdlat(dy_m=dy_m)

    N, W, S, E = [latitude + dlat, longitude - dlon, latitude - dlat, longitude + dlon]
    print("Coords box:", [N, W, S, E])
    # Round outward to 0.25 grid
    N = math.ceil(N / grid_res) * grid_res  # round north up
    S = math.floor(S / grid_res) * grid_res  # round south down
    W = math.floor(W / grid_res) * grid_res  # round west down (more negative)
    E = math.ceil(E / grid_res) * grid_res  # round east up

    area = [N, W, S, E]
    print("Rounded box:", area)
    return area


# VARIABLES INFORMATION
def GetVariableNames_PressureLevels():
    variables = [
        "u_component_of_wind",
        "v_component_of_wind",
        "vertical_velocity",
        "divergence",
        "vorticity",
        "temperature",
        "specific_humidity",
        "specific_cloud_liquid_water_content",
        "specific_cloud_ice_water_content",
        "specific_rain_water_content",
        "relative_humidity",
        "geopotential",
    ]
    return variables

def GetVariableNames_Surface():
    variables = [
        "convective_available_potential_energy",
        "convective_inhibition",
    ]
    return variables

In [95]:
##########################################################
# DOWNLOADING TRACER CAMPAIGN DATA
##########################################################

In [18]:
# coorindates information
# GETTING BOUNDING BOX centered at Houston, TX Mobile Facility (TRACER) Facility S2 ==> CSAP (C-Band Scanning ARM Precipitation Radar)
# (29.532N, 95.284W)

latitude=29.532; longitude=-95.284
area = GetCoordinates(longitude, latitude)
variables_PressureLevels = GetVariableNames_PressureLevels()
variables_Surface = GetVariableNames_Surface()

Coords box: [31.780304014796826, -97.86801827645755, 27.283695985203174, -92.69998172354246]
Rounded box: [32.0, -98.0, 27.25, -92.5]


In [16]:
###########################
# DATE ONE (DRY CASE)

In [62]:
MakeDateFolder(date_string, campaign="TRACER")

'06-05_-_06-07_2022'

In [17]:
# INFORMATION
# date information
date_string = "06-08 - 06-10 (2022)"
date_directory, date_folder = MakeDateFolder(date_string, campaign="TRACER")
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-16 09:59:52,372 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-16 09:59:52,373 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-16 09:59:53,116 INFO Request ID is 4046d41d-fbea-4eec-a218-a89d2289a3f4
2025-09-16 09:59:53,288 INFO status has been updated to accepted
2025-09-16 10:00:07,461 INFO status has been updated to running
2025-09-16 10:00:26,871 INFO status has been updated to successful


1181f8ecb1bd265500d65b9e2df940e2.nc:   0%|          | 0.00/91.3k [00:00<?, ?B/s]

In [18]:
###########################
# DATE TWO (MOIST CASE)

In [19]:
# INFORMATION
# date information
date_string = "06-30 - 07-02 (2022)"
date_directory, date_folder = MakeDateFolder(date_string, campaign="TRACER")
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-16 10:00:29,296 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-16 10:00:29,297 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-16 10:00:30,006 INFO Request ID is c7fe0117-c295-4488-91e8-faaa4809f9dd
2025-09-16 10:00:30,160 INFO status has been updated to accepted
2025-09-16 10:00:39,147 INFO status has been updated to running
2025-09-16 10:00:52,152 INFO status has been updated to successful


5fd6649d603bb648403b603e70b112c1.nc:   0%|          | 0.00/104k [00:00<?, ?B/s]

In [ ]:
###########################
# DATE THREE (INTERESTING CASE)

In [20]:
# INFORMATION
# date information
date_string = "08-11 - 08-13 (2022)"
date_directory, date_folder = MakeDateFolder(date_string, campaign="TRACER")
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-16 10:00:54,314 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-16 10:00:54,315 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-16 10:00:54,969 INFO Request ID is 742ae97c-f573-43b9-8180-4f614189c6db
2025-09-16 10:00:55,202 INFO status has been updated to accepted
2025-09-16 10:01:09,255 INFO status has been updated to running
2025-09-16 10:01:28,735 INFO status has been updated to successful


a819e22faf95cbd63d1c60d2796fed7f.nc:   0%|          | 0.00/105k [00:00<?, ?B/s]

In [78]:
##########################################################
# DOWNLOADING PRECIP CAMPAIGN DATA
##########################################################

In [103]:
# coorindates information
# GETTING BOUNDING BOX centered at Hsinchu, Taiwan PRECIP Campaign S-Pol radar moments data collected during the Prediction of Rainfall Extremes Campaign In the Pacific (PRECIP)
# (24.82N, 120.91E)

latitude=24.82; longitude=120.91
area = GetCoordinates(longitude, latitude)
variables_PressureLevels = GetVariableNames_PressureLevels()
variables_Surface = GetVariableNames_Surface()

Coords box: [27.068304014796826, 118.43288760755132, 22.571695985203174, 123.38711239244867]
Rounded box: [27.25, 118.25, 22.5, 123.5]


In [80]:
###########################
# DATE ONE (DRY CASE)

In [105]:
# INFORMATION
# date information
date_string = "06-05 - 06-07 (2022)"
date_directory, date_folder = MakeDateFolder(date_string, campaign="PRECIP")
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-23 13:46:49,717 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-23 13:46:49,717 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-23 13:46:50,112 INFO Request ID is 9d6f42b8-d0fa-4260-bfcf-5f95406c3f78
2025-09-23 13:46:50,330 INFO status has been updated to accepted
2025-09-23 13:46:59,135 INFO status has been updated to successful


f756e7d318972d1427331d44a02beed2.nc:   0%|          | 0.00/2.19M [00:00<?, ?B/s]

2025-09-23 13:47:01,835 INFO Request ID is a4f0604f-10d6-4dc0-b86a-1a666fa16934
2025-09-23 13:47:01,990 INFO status has been updated to accepted
2025-09-23 13:47:10,932 INFO status has been updated to running
2025-09-23 13:47:16,152 INFO status has been updated to accepted
2025-09-23 13:47:23,914 INFO status has been updated to successful


14ed890ff66058a1aea4b9420bb8ae4a.nc:   0%|          | 0.00/2.26M [00:00<?, ?B/s]

...

2025-09-23 13:50:43,643 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-23 13:50:43,644 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-23 13:50:44,187 INFO Request ID is ccd972c2-263d-44a4-b5a2-9c49f6903c52
2025-09-23 13:50:44,362 INFO status has been updated to accepted
2025-09-23 13:50:58,335 INFO status has been updated to successful


3f35c89b9ace6ee6024fb35a605b6451.nc:   0%|          | 0.00/82.3k [00:00<?, ?B/s]

2025-09-23 13:51:00,482 INFO Request ID is 0a0bfa06-51e3-409e-a7ed-708c6e18954f
2025-09-23 13:51:00,631 INFO status has been updated to accepted
2025-09-23 13:51:09,407 INFO status has been updated to successful


a3e0125ff805a5e8e1d8f1d12c0186e8.nc:   0%|          | 0.00/92.6k [00:00<?, ?B/s]

In [82]:
###########################
# DATE TWO (MOIST CASE)

In [104]:
# INFORMATION
# date information
date_string = "07-16 - 07-18 (2022)"
date_directory, date_folder = MakeDateFolder(date_string, campaign="PRECIP")
date_string_converted = date_string_to_range(date_string)

# running
DownloadERA5_PressureLevels(variables_PressureLevels,date_string_converted, area)
DownloadERA5_Surface(variables_Surface,date_string_converted, area)

2025-09-23 13:42:51,824 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-23 13:42:51,825 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-23 13:42:52,186 INFO Request ID is 070099c7-79de-44ad-8001-d6ab9a8a091e
2025-09-23 13:42:52,373 INFO status has been updated to accepted
2025-09-23 13:43:14,211 INFO status has been updated to successful


77de1f5170b77a9b03541ec11b7d6074.nc:   0%|          | 0.00/2.12M [00:00<?, ?B/s]

2025-09-23 13:43:17,752 INFO Request ID is 92c7c9c9-daa8-4fa6-a61f-2bc2affdaaef
2025-09-23 13:43:17,930 INFO status has been updated to accepted
2025-09-23 13:43:31,993 INFO status has been updated to successful


7c748602d275244a2ad6a404198648e9.nc:   0%|          | 0.00/2.19M [00:00<?, ?B/s]

...

2025-09-23 13:46:21,020 INFO [2025-09-03T00:00:00] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.
2025-09-23 13:46:21,022 INFO [2024-09-26T00:00:00] Watch our [Forum](https://forum.ecmwf.int/) for Announcements, news and other discussed topics.

2025-09-23 13:46:21,567 INFO Request ID is be9568ea-c270-49d6-9f40-8ecdb7cfe9c7
2025-09-23 13:46:21,729 INFO status has been updated to accepted
2025-09-23 13:46:35,842 INFO status has been updated to successful


8177e96288e0d7af02a8ef8174430f5c.nc:   0%|          | 0.00/89.2k [00:00<?, ?B/s]

2025-09-23 13:46:38,528 INFO Request ID is 26a1e84c-df8d-4800-ade0-4c9d6016be02
2025-09-23 13:46:38,714 INFO status has been updated to accepted
2025-09-23 13:46:47,508 INFO status has been updated to successful


f43a279bd5c2552c75ed868924a4ea6d.nc:   0%|          | 0.00/102k [00:00<?, ?B/s]